# RSM317 – Group Assignment 1: Cleaning and Parsing SEC Form 10-K Filings

**Team members:**
- Vladislav Glotov, Student ID: 1010946620
- Ryosuke Fukui, Student ID: 1013980694
- Andrew Ding, Student ID: 1009932218

**Goal.** For the Rotman Investment Club we build a small proof-of-concept pipeline that takes the latest
annual report (Form 10-K) of ten large US companies as plain HTML, removes the HTML tags, checks the
spelling with Norvig's spell checker, and extracts four pieces of information from every report:

1. the fiscal year end date (ISO format),
2. the text of *Item 3. Legal Proceedings*,
3. the date(s) of the signatures (ISO format),
4. the people who signed the report (without the audit firm).

The results are stored in `legalproceedings.csv`. In Section 5 we verify every extracted value against
what we read in the filings ourselves. The bonus section repeats the extraction with Beautiful Soup.

**Data.** The ten filings were downloaded from SEC EDGAR (company search → latest 10-K → open document)
and saved in this folder as `<Company>_10-K.html`. They are the only HTML files in the folder.

## 1. Loading the filings

We loop over all `*.html` files in the current directory (no file names are typed into the code).
The company name is taken from the file name, e.g. `Coca-Cola_10-K.html` → `Coca-Cola`.

In [1]:
import re
import html
import time
import unicodedata
from pathlib import Path
from collections import Counter
from datetime import datetime
from functools import lru_cache

import pandas as pd

current_directory = Path.cwd()

raw_html = {}  # company name -> raw HTML text
for html_file in sorted(current_directory.glob("*.html")):
    with html_file.open("r", encoding="utf-8") as file:
        content = file.read()
    company = html_file.stem.replace("_10-K", "").replace("_", " ")
    raw_html[company] = content
    print(f"Loaded {html_file.name}: {len(content):,} characters")

print(f"\n{len(raw_html)} filings loaded")

Loaded Alphabet_10-K.html: 2,616,614 characters
Loaded Amazon_10-K.html: 1,968,457 characters
Loaded Apple_10-K.html: 1,520,323 characters
Loaded Coca-Cola_10-K.html: 3,758,255 characters
Loaded Johnson_and_Johnson_10-K.html: 3,701,702 characters
Loaded Meta_10-K.html: 2,371,865 characters
Loaded Microsoft_10-K.html: 8,585,616 characters
Loaded Procter_and_Gamble_10-K.html: 2,599,600 characters
Loaded Tesla_10-K.html: 2,391,644 characters
Loaded Walmart_10-K.html: 2,324,096 characters

10 filings loaded


## 2. Removing the HTML tags

The filings are *inline XBRL* HTML: normal HTML plus hidden XBRL data. We clean them with regular
expressions only:

1. Delete `<script>`, `<style>` and the hidden `<ix:header>` block **including their content**.
2. Replace the end of block elements (`</p>`, `</div>`, `</tr>`, …) and `<br>` with a newline, so
   paragraphs and table rows stay on separate lines.
3. Replace the end of a table cell (`</td>`, `</th>`) with ` | `, so the columns of a table stay apart.
4. Delete all remaining tags **without inserting a space**. Some filings write small capitals as
   separate `<span>`s in the middle of a word (Alphabet: `S<span>UNDAR</span> P<span>ICHAI</span>`).
   Replacing tags with a space would give "S UNDAR P ICHAI".
5. Convert HTML entities (`&amp;`, `&#8217;`, …) with `html.unescape` and turn non-breaking spaces
   into normal spaces.
6. Tidy whitespace: one space between words, no empty lines, no empty table cells.

In [2]:
def clean_html(raw):
    "Remove HTML tags with regular expressions and return readable text."
    text = re.sub(r"(?is)<(script|style|ix:header)\b.*?</\1>", " ", raw)
    text = re.sub(r"(?i)<br\s*/?>|</(p|div|tr|li|h[1-6]|table)>", "\n", text)
    text = re.sub(r"(?i)</t[dh]>", " | ", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = html.unescape(text)
    text = text.replace("\xa0", " ").replace("​", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"( ?\| ?){2,}", " | ", text)          # several empty cells -> one separator
    text = re.sub(r"(?m)^[ |]+|[ |]+$", "", text)        # separators at the start/end of a line
    text = re.sub(r"\n\s*\n+", "\n", text)               # remove empty lines
    return text.strip()


clean_dir = current_directory / "clean_text"
clean_dir.mkdir(exist_ok=True)

clean_texts = {}
for company, raw in raw_html.items():
    clean_texts[company] = clean_html(raw)
    out_file = clean_dir / f"{company.replace(' ', '_')}_10-K.txt"
    out_file.write_text(clean_texts[company], encoding="utf-8")
    print(f"{company:22} {len(raw):>10,} -> {len(clean_texts[company]):>9,} characters  saved to {out_file.name}")

Alphabet                2,616,614 ->   348,278 characters  saved to Alphabet_10-K.txt
Amazon                  1,968,457 ->   286,487 characters  saved to Amazon_10-K.txt
Apple                   1,520,323 ->   209,418 characters  saved to Apple_10-K.txt
Coca-Cola               3,758,255 ->   606,280 characters  saved to Coca-Cola_10-K.txt


Johnson and Johnson     3,701,702 ->   423,480 characters  saved to Johnson_and_Johnson_10-K.txt
Meta                    2,371,865 ->   519,344 characters  saved to Meta_10-K.txt


Microsoft               8,585,616 ->   322,748 characters  saved to Microsoft_10-K.txt
Procter and Gamble      2,599,600 ->   309,648 characters  saved to Procter_and_Gamble_10-K.txt
Tesla                   2,391,644 ->   399,759 characters  saved to Tesla_10-K.txt
Walmart                 2,324,096 ->   363,456 characters  saved to Walmart_10-K.txt


In [3]:
# A quick look at the result: the beginning of Apple's report
print(clean_texts["Apple"][:1500])

aapl-20250927
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended September 27, 2025
or
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from to .
Commission File Number: 001-36743
Apple Inc.
(Exact name of Registrant as specified in its charter)
California | 94-2404110
(State or other jurisdiction
of incorporation or organization)
(I.R.S. Employer Identification No.)
One Apple Park Way
Cupertino, California
95014
(Address of principal executive offices) | (Zip Code)
(408) 996-1010
(Registrant’s telephone number, including area code)
Securities registered pursuant to Section 12(b) of the Act:
Title of each class | Trading symbol(s) | Name of each exchange on which registered
Common Stock, $0.00001 par value per share
AAPL | The Nasdaq Stock Market LLC
0.000% Notes due

## 3. Spell checking with Norvig's spell checker

Peter Norvig's spell checker (https://norvig.com/spell-correct.html) learns word probabilities from a
large text, the corpus. Instead of the Sherlock Holmes stories we use **Webster's Unabridged
Dictionary** from Project Gutenberg. First we remove the Gutenberg header and the licence at the end,
because they are not part of the dictionary.

In [4]:
webster_file = next(current_directory.glob("Webster*.txt"))
webster = webster_file.read_text(encoding="utf-8")
start = webster.find("*** START OF")
end = webster.find("*** END OF")
webster = webster[webster.find("\n", start) + 1 : end]


# ----- Norvig's spell checker (unchanged, only the corpus is different) -----
def words(text):
    return re.findall(r"\w+", text.lower())

WORDS = Counter(words(webster))

def P(word, N=sum(WORDS.values())):
    "Probability of `word`."
    return WORDS[word] / N

def correction(word):
    "Most probable spelling correction for word."
    return max(candidates(word), key=P)

def candidates(word):
    "Generate possible spelling corrections for word."
    return known([word]) or known(edits1(word)) or known(edits2(word)) or [word]

def known(words):
    "The subset of `words` that appear in the dictionary of WORDS."
    return set(w for w in words if w in WORDS)

def edits1(word):
    "All edits that are one edit away from `word`."
    letters    = "abcdefghijklmnopqrstuvwxyz"
    splits     = [(word[:i], word[i:])    for i in range(len(word) + 1)]
    deletes    = [L + R[1:]               for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]
    replaces   = [L + c + R[1:]           for L, R in splits if R for c in letters]
    inserts    = [L + c + R               for L, R in splits for c in letters]
    return set(deletes + transposes + replaces + inserts)

def edits2(word):
    "All edits that are two edits away from `word`."
    return (e2 for e1 in edits1(word) for e2 in edits1(e1))


print(f"Webster's corpus: {len(WORDS):,} different words, {sum(WORDS.values()):,} words in total")
print("Most common words:", WORDS.most_common(10))

Webster's corpus: 211,279 different words, 4,734,121 words in total
Most common words: [('a', 209430), ('the', 194699), ('of', 177750), ('to', 153687), ('or', 111735), ('defn', 102760), ('n', 72833), ('in', 70103), ('and', 62310), ('as', 56331)]


### 3.1 A naive first run

We apply Norvig's tokenizer to every cleaned filing and count how many of the *different* words are
not in Webster's dictionary.

In [5]:
naive_rows = []
for company, text in clean_texts.items():
    unique_tokens = set(words(text))
    unknown = [w for w in unique_tokens if w not in WORDS]
    naive_rows.append({"company": company, "unique_tokens": len(unique_tokens), "unknown_naive": len(unknown)})

naive_df = pd.DataFrame(naive_rows)
naive_df

,company,unique_tokens,unknown_naive
0,Alphabet,4317,873
1,Amazon,3830,821
2,Apple,3346,687
3,Coca-Cola,5348,1299
4,Johnson and Johnson,5104,1296
5,Meta,4580,803
6,Microsoft,4244,907
7,Procter and Gamble,4053,876
8,Tesla,4602,892
9,Walmart,4753,982


In [6]:
# What does the checker suggest for some typical 10-K words that are spelled correctly?
test_words = ["cybersecurity", "smartphone", "iphone", "online", "website", "ebitda",
              "blockchain", "nasdaq", "subsidiaries", "fiscal", "litigation", "covid"]

start_time = time.time()
for w in test_words:
    print(f"{w:15} in Webster? {str(w in WORDS):5}  ->  {correction(w)}")
print(f"\n{len(test_words)} words took {time.time() - start_time:.1f} seconds")

cybersecurity   in Webster? False  ->  cybersecurity
smartphone      in Webster? False  ->  smartphone
iphone          in Webster? False  ->  phone
online          in Webster? False  ->  unline
website         in Webster? False  ->  webster


ebitda          in Webster? False  ->  bita
blockchain      in Webster? False  ->  blockchain
nasdaq          in Webster? False  ->  nasal
subsidiaries    in Webster? True   ->  subsidiaries
fiscal          in Webster? True   ->  fiscal
litigation      in Webster? True   ->  litigation
covid           in Webster? False  ->  coid

12 words took 0.2 seconds


In [7]:
# How long does the original checker need for ALL unknown words of one report (Apple)?
apple_unknown = sorted(w for w in set(words(clean_texts["Apple"])) if w not in WORDS)
start_time = time.time()
apple_corrections = {w: correction(w) for w in apple_unknown}
seconds = time.time() - start_time
changed = sum(1 for w, c in apple_corrections.items() if w != c)
print(f"{len(apple_unknown)} unknown words in Apple's 10-K, checked in {seconds:.1f} seconds; {changed} would be changed")
print("Some of them:", list(apple_corrections.items())[:15])

687 unknown words in Apple's 10-K, checked in 4.2 seconds; 641 would be changed
Some of them: [('00001', '000'), ('00042', '000'), ('001', '00'), ('003', '00'), ('004', '00'), ('007', '00'), ('010', '10'), ('014', '14'), ('016', '16'), ('020', '20'), ('023', '23'), ('024', '24'), ('027', '27'), ('03', '3'), ('030', '30')]


### 3.2 Challenges of this spell checker

The outputs above show the main problems:

1. **The corpus is a dictionary, not normal text.** Norvig's checker picks the candidate with the
   highest frequency `P(word)`. In a dictionary, a word's frequency says how often it is used *in
   definitions* (words like "of", "the", "or", "a", "defn" dominate), not how common it is in real
   language, so the ranking of candidates is often poor.
2. **Old vocabulary.** Webster's 1913 edition has no modern words (e.g. *online*, *website*,
   *smartphone*, *cybersecurity*), so correct words are treated as typos and "corrected" into
   unrelated old words.
3. **Names, brands and acronyms.** Company names, products, people and tickers (*iPhone*, *Nasdaq*,
   *EBITDA*, *GAAP*) are not in any dictionary.
4. **Numbers and identifiers.** `\w+` keeps numbers as "words", and the checker happily "corrects"
   them (`010` → `10`, `00001` → `000` in Apple's report).
5. **Inflected forms.** A dictionary lists base forms; many inflected forms are missing (in our
   results e.g. *collaborating*, *amortizing*).
6. **Speed.** `edits2` creates hundreds of thousands of candidates for a long word. For a few words
   this is fast, but the unknown words of one report already take several seconds (Apple above), and
   the Investment Club's full archive (about 2 TB) would take far too long this way.
7. **A wrong correction is worse than no correction.** Changing a correct finance term damages the
   text we want to analyse.

### 3.3 How we address them

* We only check **lower-case words made of letters**, with at least 3 letters. Numbers, acronyms in
  capitals and mixed-case brand names (*iPhone*) are skipped. A capitalised word that never appears in
  lower case in the same filing is treated as a proper noun and skipped.
* We add a **domain vocabulary**: every word that appears in at least 3 of the 10 filings is accepted
  as correct (a real typo is very unlikely to appear in three different companies' reports).
* We accept **inflected forms** if the word without a common ending (`s, es, ed, d, ing, ly, er, 's`,
  `ies → y`, `ing → e`) is known.
* For **speed** we check every different word only once, cache results with `lru_cache`, and skip
  `edits2` for words longer than 10 letters.
* We use the checker to **flag** possible errors and do **not** overwrite the clean text files.

In [8]:
# Domain vocabulary: words used in at least 3 of the 10 filings
doc_freq = Counter()
for text in clean_texts.values():
    doc_freq.update(set(words(text)))
domain_vocab = {w for w, n in doc_freq.items() if n >= 3}
VOCAB = set(WORDS) | domain_vocab


def is_known(word):
    "A word is known if it, or the word without a common ending, is in VOCAB."
    if word in VOCAB:
        return True
    for ending in ("'s", "s", "es", "ed", "d", "ing", "ly", "er"):
        if word.endswith(ending) and word[: -len(ending)] in VOCAB:
            return True
    if word.endswith("ing") and word[:-3] + "e" in VOCAB:      # collaborating -> collaborate
        return True
    return word.endswith("ies") and word[:-3] + "y" in VOCAB


def tokens_to_check(text):
    "Different lower-case words worth checking (no numbers, acronyms, brands or proper nouns)."
    tokens = re.findall(r"[A-Za-z]+(?:'[a-z]+)?", text.replace("’", "'"))
    lower_forms = {t for t in tokens if t.islower()}
    keep = set()
    for t in tokens:
        if len(t) < 3 or t.isupper():                 # short words and acronyms (GAAP, AWS)
            continue
        if not t.islower() and not t.istitle():       # mixed case brands (iPhone, PayPal)
            continue
        if t.istitle() and t.lower() not in lower_forms:   # only ever capitalised -> proper noun
            continue
        keep.add(t.lower())
    return keep


@lru_cache(maxsize=None)
def fast_correction(word):
    "Norvig's correction with a cache and without edits2 for long words."
    cands = known([word]) or known(edits1(word))
    if not cands and len(word) <= 10:
        cands = known(edits2(word))
    return max(cands, key=P) if cands else word


improved_rows, examples = [], []
start_time = time.time()
for company, text in clean_texts.items():
    flagged = sorted(w for w in tokens_to_check(text) if not is_known(w))
    improved_rows.append({"company": company, "unknown_improved": len(flagged)})
    for w in flagged[:5]:
        examples.append({"company": company, "flagged word": w, "suggestion": fast_correction(w)})

spell_df = naive_df.merge(pd.DataFrame(improved_rows), on="company")
print(f"Improved check took {time.time() - start_time:.1f} seconds")
spell_df

Improved check took 0.6 seconds


,company,unique_tokens,unknown_naive,unknown_improved
0,Alphabet,4317,873,50
1,Amazon,3830,821,34
2,Apple,3346,687,30
3,Coca-Cola,5348,1299,123
4,Johnson and Johnson,5104,1296,134
5,Meta,4580,803,58
6,Microsoft,4244,907,52
7,Procter and Gamble,4053,876,35
8,Tesla,4602,892,85
9,Walmart,4753,982,65


In [9]:
# Examples of words that are still flagged, with the checker's suggestion
pd.DataFrame(examples)

,company,flagged word,suggestion
0,Alphabet,abc,ac
1,Alphabet,algorithmic,algorithmic
2,Alphabet,blocklisting,blocklisting
3,Alphabet,blog,blow
4,Alphabet,blogs,blows
5,Amazon,amzn,amen
6,Amazon,audiobook,audiobook
7,Amazon,audiobooks,audiobooks
8,Amazon,benchmark,benchmark
9,Amazon,broadband,broadband


### 3.4 What is left

The filters and the domain vocabulary reduce the number of flagged words per report from roughly
700–1,300 to roughly 30–135 (table above). Looking at the remaining words, almost none of them are real
spelling mistakes. Most are correct modern or technical words that Webster's 1913 dictionary does not
know (*blog*, *bandwidth*, *bitcoin*, *audio*, drug names in the J&J report such as *amivantamab*),
product names written in lower case, and pieces of web addresses (*aspx*, *atmeta*).
The suggestions show why we do not let the checker overwrite the text: *audio* → *audit*,
*airport* → *import*, *activism* → *actinism* and *bitcoin* → *biscotin* would all damage the report.
Coca-Cola and J&J have the most flagged words; for J&J many of them are drug names, and Coca-Cola's
list includes ingredient names such as *acesulfame* next to ordinary modern words (*affordability*).
**Conclusion:** with a 1913 dictionary as corpus, Norvig's checker is useful to *flag* unusual words,
but for professionally edited 10-K filings it should not correct them automatically. A better corpus
would be a large collection of recent 10-K filings themselves.

## 4. Extracting the information

All functions below work on the cleaned text and use regular expressions. Each returns an empty
string if it finds nothing, so one unusual filing cannot stop the whole loop.

**Dates.** A date like "September 27, 2025" is matched with the pattern `DATE` and converted to ISO
format (`2025-09-27`) with `datetime.strptime`.

In [10]:
MONTH = r"(?:January|February|March|April|May|June|July|August|September|October|November|December)"
DATE = MONTH + r"\s+\d{1,2}\s*,\s*\d{4}"


def to_iso(date_str):
    "'September 27 , 2025' -> '2025-09-27'"
    date_str = re.sub(r"\s*,\s*", ", ", re.sub(r"\s+", " ", date_str.strip()))
    return datetime.strptime(date_str, "%B %d, %Y").date().isoformat()


print(to_iso("September 27 , 2025"), to_iso("JUNE 30, 2026"))

2025-09-27 2026-06-30


**1. Fiscal year end.** The cover page always says "For the fiscal year ended *date*". We take the
first match, which is on the cover page.

In [11]:
def get_fiscal_year_end(text):
    match = re.search(r"(?i)fiscal\s+year\s+ended\s*:?\s*(" + DATE + ")", text)
    return to_iso(match.group(1)) if match else ""

**2. Item 3. Legal Proceedings.** The heading appears twice: first in the table of contents, then
as the real section heading. We therefore take the **last** line that starts with
"Item 3 … Legal Proceedings" and cut the text at the next line that starts with "Item 4".

In [12]:
def get_item3(text):
    starts = list(re.finditer(r"(?im)^\s*item\s*3\s*[.:]?\s*(?:\|\s*)?legal\s+proceedings\.?", text))
    if not starts:
        return ""
    start = starts[-1].end()
    end = re.search(r"(?im)^\s*item\s*4\b", text[start:])
    section = text[start : start + end.start()] if end else text[start:]
    return section.lstrip(" .|\n").rstrip(" |\n")

**3. and 4. Signatures.** The signature page starts with a line that only says "SIGNATURES"
(again, the last one, because the first is in the table of contents). Some companies put the exhibit
index after it, so we stop there.

* **Dates:** every date in the section, plus the form "28th day of January 2026" (Meta). The footer
  "Form 10-K for the Fiscal Year Ended …" (Walmart) is not a signature date, so we remove it first.
* **Signers:** every name written after `/s/` (some filings write `/ S /`).
  * Many filings write the `/s/` name in capital letters and print the name normally on the next line
    (e.g. `/s/ CHRISTINE M. MCCARTHY` and `(Christine M. McCarthy)`). We look for that printed version in
    the section, so we get "McCarthy" instead of `title()`'s "Mccarthy". Accents are ignored when
    comparing (`CHAVEZ` = `Chávez`).
  * Some directors sign through an attorney-in-fact: their name is marked with `*` instead of `/s/`
    (Coca-Cola). We therefore also take every name that starts a line (or a table cell) and is directly
    followed by a line saying "Director".
  * A comma before "Jr." is removed (otherwise it would break our comma-separated list), the audit firm
    (contains "LLP") is dropped, and each person is listed only once.

In [13]:
def get_signature_section(text):
    heads = list(re.finditer(r"(?im)^\s*signatures\s*$", text))
    if not heads:
        return ""
    section = text[heads[-1].start():]
    exhibit = re.search(r"(?im)^\s*exhibit\s+index", section)
    return section[: exhibit.start()] if exhibit else section


def get_signature_dates(section):
    section = re.sub(r"(?i)fiscal\s+year\s+ended\s*" + DATE, " ", section)
    found = {to_iso(d) for d in re.findall(DATE, section, flags=re.I)}
    long_form = r"(\d{1,2})(?:st|nd|rd|th)\s+day\s+of\s+(" + MONTH + r")\s*,?\s*(\d{4})"
    for day, month, year in re.findall(long_form, section, flags=re.I):
        found.add(to_iso(f"{month} {day}, {year}"))
    return ", ".join(sorted(found))


def plain_letters(name):
    "Lower case without accents and brackets: 'Chávez' -> 'chavez', so names can be compared."
    name = unicodedata.normalize("NFKD", name.strip(" ()"))
    return "".join(ch for ch in name if not unicodedata.combining(ch)).lower()


def get_signers(section):
    names = re.findall(r"(?i)/\s*s\s*/\s*([^|\n/]+)", section)
    # directors who signed through an attorney-in-fact: "Name" followed by a line "Director"
    names += re.findall(r"(?m)(?:^|\| )([A-Z][\w.'-]*(?: [A-Z][\w.'-]*){1,3})\nDirector\b", section)
    # all lines / table cells of the section, used to find the normally printed version of a name
    cells = [c.strip(" ()") for c in re.split(r"\n|\|", section)]

    signers = []
    for name in names:
        name = name.strip(" ,*")
        name = re.sub(r",?\s+(Jr\.|Sr\.|II|III)$", r" \1", name, flags=re.I)
        if not name or "LLP" in name.upper():
            continue
        if name.isupper():
            printed = [c for c in cells if not c.isupper() and plain_letters(c) == plain_letters(name)]
            name = printed[0] if printed else name.title()
        if plain_letters(name) not in [plain_letters(s) for s in signers]:
            signers.append(name)
    return ", ".join(signers)

Now we run all functions for every company. As suggested, we collect the results in a dictionary of
lists first and turn it into a DataFrame at the end.

In [14]:
results = {"company": [], "fiscal_year_end": [], "legal_proceedings": [], "signature_dates": [], "signers": []}

for company, text in clean_texts.items():
    signature_section = get_signature_section(text)
    results["company"].append(company)
    results["fiscal_year_end"].append(get_fiscal_year_end(text))
    results["legal_proceedings"].append(get_item3(text))
    results["signature_dates"].append(get_signature_dates(signature_section))
    results["signers"].append(get_signers(signature_section))

df = pd.DataFrame(results)
df.to_csv("legalproceedings.csv", index=False)

pd.set_option("display.max_colwidth", 120)
df.assign(legal_proceedings=df["legal_proceedings"].str.len()).rename(
    columns={"legal_proceedings": "item3_length"})

,company,fiscal_year_end,item3_length,signature_dates,signers
0,Alphabet,2025-12-31,246,2026-02-04,"Sundar Pichai, Anat Ashkenazi, Amie Thuener O'Toole, Frances H. Arnold, Sergey Brin, R. Martin Chávez, L. John Doerr..."
1,Amazon,2025-12-31,130,2026-02-05,"Andrew R. Jassy, Brian T. Olsavsky, Shelley L. Reynolds, Jeffrey P. Bezos, Keith B. Alexander, Edith W. Cooper, Jami..."
2,Apple,2025-09-27,5372,2025-10-31,"Kevan Parekh, Timothy D. Cook, Chris Kondo, Wanda Austin, Alex Gorsky, Andrea Jung, Arthur D. Levinson, Monica Lozan..."
3,Coca-Cola,2025-12-31,22067,2026-02-20,"James Quincey, John Murphy, Erin L. May, Jennifer D. Manning, Herb Allen, Christopher C. Davis, Bela Bajaria, Caroly..."
4,Johnson and Johnson,2025-12-28,218,2026-02-11,"J. Duato, J. J. Wolk, R. J. Decker Jr., M. C. Beckerle, J. A. Doudna, M. A. Hewson, P. A. Johnson, H. Joly, M. B. Mc..."
5,Meta,2025-12-31,38480,2026-01-28,"Susan Li, Mark Zuckerberg, Aaron Anderson, Peggy Alford, Marc L. Andreessen, John Arnold, Patrick Collison, John Elk..."
6,Microsoft,2026-06-30,175,2026-07-29,"Alice L. Jolla, Satya Nadella, Carmine Di Sibio, Reid G. Hoffman, Hugh F. Johnston, Teri L. List, Catherine MacGrego..."
7,Procter and Gamble,2026-06-30,1852,2026-08-04,"Shailesh Jejurikar, Andre Schulten, Matthew W. Janzaruk, B. Marc Allen, Craig Arnold, Brett Biggs, Sheila Bonini, Am..."
8,Tesla,2025-12-31,200,2026-01-28,"Elon Musk, Vaibhav Taneja, Robyn Denholm, Ira Ehrenpreis, Joseph Gebbia, Jack Hartung, James Murdoch, Kimbal Musk, J..."
9,Walmart,2026-01-31,5390,2026-03-13,"John R. Furner, Gregory B. Penner, John David Rainey, Dwayne M. Milum, Cesar Conde, Timothy P. Flynn, Sarah Friar, C..."


## 5. Verification

### (a) Start and end of "Item 3. Legal Proceedings"

For every company we print the first and last 200 characters of the extracted section and check
automatically that no "Item 4" heading was included.

In [15]:
for company, text in zip(df["company"], df["legal_proceedings"]):
    has_item4 = re.search(r"(?im)^\s*item\s*4\b", text) is not None
    print("=" * 100)
    print(f"{company}  ({len(text):,} characters)   contains an 'Item 4' heading: {has_item4}")
    print("FIRST 200:", repr(text[:200]))
    print("LAST 200: ", repr(text[-200:]))

Alphabet  (246 characters)   contains an 'Item 4' heading: False
FIRST 200: 'For a description of our material pending legal proceedings, see Legal Matters in Note 10 of the Notes to Consolidated Financial Statements included in Part II, Item 8 of this Annual Report on Form 10'
LAST 200:  'l proceedings, see Legal Matters in Note 10 of the Notes to Consolidated Financial Statements included in Part II, Item 8 of this Annual Report on Form 10-K, which is incorporated herein by reference.'
Amazon  (130 characters)   contains an 'Item 4' heading: False
FIRST 200: 'See Item 8 of Part II, “Financial Statements and Supplementary Data — Note 7 — Commitments and Contingencies — Legal Proceedings.”'
LAST 200:  'See Item 8 of Part II, “Financial Statements and Supplementary Data — Note 7 — Commitments and Contingencies — Legal Proceedings.”'
Apple  (5,372 characters)   contains an 'Item 4' heading: False
FIRST 200: 'Digital Markets Act Investigations\nOn March 25, 2024, the Commission announced 

### (b) Check table

The values in `manual` were read by hand from the cover page and the signature pages of each filing:
the fiscal year end, the signature date(s) and every person who signed, written as printed in the
filing. Signers are compared as a set of names: the order does not matter, but spelling, accents and
capitalisation do.

In [16]:
manual = {
    "Apple": {
        "fiscal_year_end": "2025-09-27", "signature_dates": "2025-10-31",
        "signers": "Kevan Parekh, Timothy D. Cook, Chris Kondo, Wanda Austin, Alex Gorsky, Andrea Jung, "
                   "Arthur D. Levinson, Monica Lozano, Ronald D. Sugar, Susan L. Wagner"},
    "Microsoft": {
        "fiscal_year_end": "2026-06-30", "signature_dates": "2026-07-29",
        "signers": "Alice L. Jolla, Satya Nadella, Carmine Di Sibio, Reid G. Hoffman, Hugh F. Johnston, "
                   "Teri L. List, Catherine MacGregor, Mark A. L. Mason, Sandra E. Peterson, Penny S. Pritzker, "
                   "John David Rainey, Charles W. Scharf, John W. Stanton, Emma N. Walmsley, Amy E. Hood"},
    "Amazon": {
        "fiscal_year_end": "2025-12-31", "signature_dates": "2026-02-05",
        "signers": "Andrew R. Jassy, Brian T. Olsavsky, Shelley L. Reynolds, Jeffrey P. Bezos, Keith B. Alexander, "
                   "Edith W. Cooper, Jamie S. Gorelick, Daniel P. Huttenlocher, Andrew Y. Ng, Indra K. Nooyi, "
                   "Jonathan J. Rubinstein, Brad D. Smith, Patricia Q. Stonesifer, Wendell P. Weeks"},
    "Tesla": {
        "fiscal_year_end": "2025-12-31", "signature_dates": "2026-01-28",
        "signers": "Elon Musk, Vaibhav Taneja, Robyn Denholm, Ira Ehrenpreis, Joseph Gebbia, Jack Hartung, "
                   "James Murdoch, Kimbal Musk, JB Straubel, Kathleen Wilson-Thompson"},
    "Meta": {
        "fiscal_year_end": "2025-12-31", "signature_dates": "2026-01-28",
        "signers": "Susan Li, Mark Zuckerberg, Aaron Anderson, Peggy Alford, Marc L. Andreessen, John Arnold, "
                   "Patrick Collison, John Elkann, Andrew W. Houston, Nancy Killefer, Robert M. Kimmitt, "
                   "Charles Songhurst, Hock E. Tan, Tracey T. Travis, Dana White, Tony Xu"},
    "Alphabet": {
        "fiscal_year_end": "2025-12-31", "signature_dates": "2026-02-04",
        "signers": "Sundar Pichai, Anat Ashkenazi, Amie Thuener O'Toole, Frances H. Arnold, Sergey Brin, "
                   "R. Martin Chávez, L. John Doerr, Roger W. Ferguson Jr., John L. Hennessy, Larry Page, "
                   "K. Ram Shriram, Robin L. Washington"},
    "Johnson and Johnson": {
        "fiscal_year_end": "2025-12-28", "signature_dates": "2026-02-11",
        "signers": "J. Duato, J. J. Wolk, R. J. Decker Jr., M. C. Beckerle, J. A. Doudna, M. A. Hewson, "
                   "P. A. Johnson, H. Joly, M. B. McClellan, J. G. Morikis, D. E. Pinto, M. A. Weinberger, "
                   "N. Y. West, E. A. Woods"},
    "Coca-Cola": {
        "fiscal_year_end": "2025-12-31", "signature_dates": "2026-02-20",
        "signers": "James Quincey, John Murphy, Erin L. May, Herb Allen, Christopher C. Davis, Bela Bajaria, "
                   "Carolyn Everson, Ana Botín, Thomas S. Gayner, Maria Elena Lagomasino, Caroline J. Tsay, "
                   "Max Levchin, David B. Weinberg, Amity Millhiser, Jennifer D. Manning"},
    "Procter and Gamble": {
        "fiscal_year_end": "2026-06-30", "signature_dates": "2026-08-04",
        "signers": "Shailesh Jejurikar, Andre Schulten, Matthew W. Janzaruk, B. Marc Allen, Craig Arnold, "
                   "Brett Biggs, Sheila Bonini, Amy L. Chang, Joseph Jimenez, Christopher J. Kempczinski, "
                   "Debra L. Lee, Christine M. McCarthy, Ashley McEvoy, Robert J. Portman, Rajesh Subramaniam"},
    "Walmart": {
        "fiscal_year_end": "2026-01-31", "signature_dates": "2026-03-13",
        "signers": "John R. Furner, Gregory B. Penner, John David Rainey, Dwayne M. Milum, Cesar Conde, "
                   "Timothy P. Flynn, Sarah Friar, Carla A. Harris, Thomas W. Horton, Marissa A. Mayer, "
                   "C. Douglas McMillon, Shishir Mehrotra, Robert E. Moritz Jr., Brian Niccol, "
                   "Randall L. Stephenson, Steuart L. Walton"},
}


def name_set(names):
    return {re.sub(r"\s+", " ", n).strip() for n in names.split(",") if n.strip()}


def same_names(a, b):
    return name_set(a) == name_set(b)


check_rows = []
for _, row in df.iterrows():
    truth = manual[row["company"]]
    for field in ["fiscal_year_end", "signature_dates", "signers"]:
        ok = same_names(row[field], truth[field]) if field == "signers" else row[field] == truth[field]
        check_rows.append({"company": row["company"], "field": field, "extracted": row[field],
                           "manual (read by us)": truth[field], "match": "✔" if ok else "✘ MISMATCH"})

check_df = pd.DataFrame(check_rows)
pd.set_option("display.max_colwidth", None)   # show the full signer lists
print(check_df["match"].value_counts().to_string())
check_df

match
✔    30


,company,field,extracted,manual (read by us),match
0,Alphabet,fiscal_year_end,2025-12-31,2025-12-31,✔
1,Alphabet,signature_dates,2026-02-04,2026-02-04,✔
2,Alphabet,signers,"Sundar Pichai, Anat Ashkenazi, Amie Thuener O'Toole, Frances H. Arnold, Sergey Brin, R. Martin Chávez, L. John Doerr, Roger W. Ferguson Jr., John L. Hennessy, Larry Page, K. Ram Shriram, Robin L. Washington","Sundar Pichai, Anat Ashkenazi, Amie Thuener O'Toole, Frances H. Arnold, Sergey Brin, R. Martin Chávez, L. John Doerr, Roger W. Ferguson Jr., John L. Hennessy, Larry Page, K. Ram Shriram, Robin L. Washington",✔
3,Amazon,fiscal_year_end,2025-12-31,2025-12-31,✔
4,Amazon,signature_dates,2026-02-05,2026-02-05,✔
5,Amazon,signers,"Andrew R. Jassy, Brian T. Olsavsky, Shelley L. Reynolds, Jeffrey P. Bezos, Keith B. Alexander, Edith W. Cooper, Jamie S. Gorelick, Daniel P. Huttenlocher, Andrew Y. Ng, Indra K. Nooyi, Jonathan J. Rubinstein, Brad D. Smith, Patricia Q. Stonesifer, Wendell P. Weeks","Andrew R. Jassy, Brian T. Olsavsky, Shelley L. Reynolds, Jeffrey P. Bezos, Keith B. Alexander, Edith W. Cooper, Jamie S. Gorelick, Daniel P. Huttenlocher, Andrew Y. Ng, Indra K. Nooyi, Jonathan J. Rubinstein, Brad D. Smith, Patricia Q. Stonesifer, Wendell P. Weeks",✔
6,Apple,fiscal_year_end,2025-09-27,2025-09-27,✔
7,Apple,signature_dates,2025-10-31,2025-10-31,✔
8,Apple,signers,"Kevan Parekh, Timothy D. Cook, Chris Kondo, Wanda Austin, Alex Gorsky, Andrea Jung, Arthur D. Levinson, Monica Lozano, Ronald D. Sugar, Susan L. Wagner","Kevan Parekh, Timothy D. Cook, Chris Kondo, Wanda Austin, Alex Gorsky, Andrea Jung, Arthur D. Levinson, Monica Lozano, Ronald D. Sugar, Susan L. Wagner",✔
9,Coca-Cola,fiscal_year_end,2025-12-31,2025-12-31,✔


In [17]:
# Details of every mismatch
for _, r in check_df[check_df["match"] != "✔"].iterrows():
    if r["field"] == "signers":
        missing = sorted(name_set(r["manual (read by us)"]) - name_set(r["extracted"]))
        extra = sorted(name_set(r["extracted"]) - name_set(r["manual (read by us)"]))
        print(f"{r['company']} signers  missing: {missing}  extra: {extra}")
    else:
        print(f"{r['company']} {r['field']}: extracted {r['extracted']!r}, filing says {r['manual (read by us)']!r}")

**Result of (a).** No section contains an "Item 4" heading, and we checked in the clean text that
the line directly after each extracted section is the "Item 4. Mine Safety Disclosures" heading.

| Company | Starts with | Ends with |
|---|---|---|
| Alphabet | reference to Note 10 (one sentence) | "…incorporated herein by reference." ✔ |
| Amazon | "See Item 8 of Part II, … Note 7 …" (one sentence) | same sentence ✔ |
| Apple | sub-heading "Digital Markets Act Investigations" | last paragraph on possible losses ✔ |
| Coca-Cola | "The Company is involved in various legal proceedings…" | last pending case ("strong defenses to the claims.") ✔ |
| Johnson and Johnson | reference to Note 19 (one sentence) | same sentence ✔ |
| Meta | "As a multinational company…" | "…legal proceedings and disputes in the future." ✔ |
| Microsoft | reference to Note 14 (one sentence) | same sentence ✔ |
| Procter and Gamble | "The Company is subject, from time to time…" | reference to Risk Factors (Item 1A) ✔ |
| Tesla | reference to Note 13 (one sentence) | same sentence ✔ |
| Walmart | sub-heading "I. SUPPLEMENTAL INFORMATION" | last paragraph on possible adverse effects ✔ |

Five companies only refer to a note in the financial statements; the actual case descriptions are in
Item 8. The long sections of Coca-Cola and Meta still contain page numbers and "Table of Contents"
lines from page breaks; these are part of the HTML text and do not change where the section starts or ends.

### (c) Explanation of the mismatches

**Result:** after the fixes below, all 30 values match (30 ✔, no ✘).

**Coca-Cola – signers (was a ✘ in our first version).** At first our function only found James
Quincey, John Murphy, Erin L. May and Jennifer D. Manning, but not the 11 directors. In Coca-Cola's
filing the directors did not sign with `/s/`: their names are marked with `*`, and Jennifer D. Manning
signed for all of them as attorney-in-fact ("*By: /s/ JENNIFER D. MANNING … Attorney-in-fact"). We
fixed this with a general rule (not one written only for Coca-Cola): a name at the start of a line or
table cell that is directly followed by the title "Director" is also a signer. For the other nine
companies this rule only finds directors who are already in the list, so their results do not change.
A remaining limitation: an attorney-in-fact signer with a title other than "Director" in a layout like
Coca-Cola's would still be missed.

**Capitalisation of names (was hidden by our first check table).** Our first version turned capital-
letter names with `title()` into "Macgregor", "Mccarthy", "Mcevoy" and "Chavez", and our first check
table compared names in lower case, so it showed ✔ although the CSV was wrong. We now take the
normally printed name from the filing (see Section 4) and compare names case-sensitively.

**Problems we found and fixed while developing** (each check was re-run with the fix removed to
confirm it matters):

* **Small capitals split by tags (Alphabet, Microsoft).** When tags were replaced by a space, names
  came out as "S Undar P Ichai" and Microsoft's "SIGNATURES" heading became "S IGNATURES", so its
  signature section was not found and dozens of exhibit dates were reported. Fix: remove inline tags
  without a space (Section 2). With spaces, the Item 3 text of five companies also contained extra
  spaces inside words and before punctuation (e.g. Tesla "Contingencies , to").
* **Table of contents (9 of 10 companies).** The first "Item 3 Legal Proceedings" match is the table
  of contents, so the extracted "section" was just a page number. Fix: use the last match.
* **Walmart signature date.** Each Walmart signature page has the footer "Form 10-K for the Fiscal
  Year Ended January 31, 2026", which added 2026-01-31 as a signature date. Fix: remove "fiscal year
  ended + date" before collecting dates.
* **Johnson & Johnson signature dates.** J&J's exhibit index comes after the signatures and contains
  many old dates. Fix: cut the signature section at "Exhibit index".
* **Walmart "Robert E. Moritz, Jr.".** The comma inside the name would split one person into two
  entries in our comma-separated list. Fix: remove the comma before "Jr.".
* **Audit firm.** Every report contains the auditor's "/s/ … LLP" signature, but always in the
  auditor's report, never in the Signatures section. Restricting the search to the Signatures section
  keeps it out; the "LLP" filter is an extra safety check.

## Bonus: Beautiful Soup

We repeat the cleaning with Beautiful Soup and run the **same** extraction functions on its text.
Beautiful Soup parses the HTML properly, so we do not need our own tag regexes. We remove the same
hidden elements (`script`, `style`, `ix:header`) and use `get_text("\n")`, which puts a newline
between all pieces of text.

In [18]:
from bs4 import BeautifulSoup


def clean_with_bs4(raw):
    soup = BeautifulSoup(raw, "html.parser")
    for tag in soup(["script", "style", "ix:header"]):
        tag.decompose()
    text = soup.get_text("\n").replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()


def plain(text):
    "Item 3 text without table separators and extra spaces, so both versions can be compared."
    return " ".join(text.replace("|", " ").split())


bonus_rows = []
start_time = time.time()
regex_results = df.set_index("company")
for company, raw in raw_html.items():
    text_bs = clean_with_bs4(raw)
    section_bs = get_signature_section(text_bs)
    bs4_values = {
        "fiscal_year_end": get_fiscal_year_end(text_bs),
        "legal_proceedings": get_item3(text_bs),
        "signature_dates": get_signature_dates(section_bs),
        "signers": get_signers(section_bs),
    }
    for field, bs_value in bs4_values.items():
        rx_value = regex_results.loc[company, field]
        if field == "legal_proceedings":
            agree = plain(rx_value) == plain(bs_value)
            rx_ok = bs_ok = "see (a)"
            rx_show, bs_show = f"{len(plain(rx_value)):,} chars", f"{len(plain(bs_value)):,} chars"
        else:
            truth = manual[company][field]
            same = same_names if field == "signers" else (lambda a, b: a == b)
            agree = same(rx_value, bs_value)
            rx_ok, bs_ok = same(rx_value, truth), same(bs_value, truth)
            rx_show, bs_show = rx_value, bs_value
        bonus_rows.append({"company": company, "field": field, "regex": rx_show, "bs4": bs_show,
                           "agree": "✔" if agree else "✘", "regex correct": rx_ok, "bs4 correct": bs_ok})

bonus_df = pd.DataFrame(bonus_rows)
print(f"Beautiful Soup version took {time.time() - start_time:.0f} seconds")
print(bonus_df["agree"].value_counts().to_string())
bonus_df[bonus_df["agree"] == "✘"]

Beautiful Soup version took 2 seconds
agree
✔    33
✘     7


,company,field,regex,bs4,agree,regex correct,bs4 correct
3,Alphabet,signers,"Sundar Pichai, Anat Ashkenazi, Amie Thuener O'Toole, Frances H. Arnold, Sergey Brin, R. Martin Chávez, L. John Doerr, Roger W. Ferguson Jr., John L. Hennessy, Larry Page, K. Ram Shriram, Robin L. Washington","S, A, F, R. M, L. J, R, J, L, K. R",✘,True,False
13,Coca-Cola,legal_proceedings,"22,067 chars","22,087 chars",✘,see (a),see (a)
21,Meta,legal_proceedings,"38,480 chars","38,526 chars",✘,see (a),see (a)
26,Microsoft,signature_dates,2026-07-29,"2009-05-18, 2010-09-27, 2011-02-08, 2012-11-07, 2013-05-02, 2013-12-06, 2014-02-03, 2015-02-12, 2015-11-03, 2016-08-08, 2016-09-19, 2017-02-06, 2017-05-26, 2020-06-01, 2020-08-10, 2021-03-17, 2021-09-14, 2022-07-01, 2023-09-26, 2023-10-13, 2023-10-27, 2023-11-06, 2024-01-19, 2024-03-08, 2024-05-01, 2024-06-30, 2024-09-16, 2024-09-30, 2024-11-21, 2024-12-03, 2024-12-12, 2025-02-20, 2025-03-11, 2025-03-13, 2025-05-01, 2025-05-15, 2025-06-10, 2025-06-12, 2025-06-30, 2025-08-21, 2025-09-11, 2025-09-15, 2025-11-20, 2025-12-02, 2025-12-11, 2026-02-19, 2026-03-10, 2026-03-12, 2026-04-01, 2026-04-30, 2026-05-01, 2026-05-21, 2026-05-31, 2026-06-01, 2026-06-10, 2026-06-11, 2026-06-30, 2026-07-23, 2026-07-29, 2026-08-20, 2026-08-31, 2026-09-10, 2026-09-14, 2026-09-30, 2026-12-08, 2026-12-31, 2027-02-28, 2027-03-05, 2027-03-31, 2027-06-30",✘,True,False
27,Microsoft,signers,"Alice L. Jolla, Satya Nadella, Carmine Di Sibio, Reid G. Hoffman, Hugh F. Johnston, Teri L. List, Catherine MacGregor, Mark A. L. Mason, Sandra E. Peterson, Penny S. Pritzker, John David Rainey, Charles W. Scharf, John W. Stanton, Emma N. Walmsley, Amy E. Hood","d, a, S, c, R, H, T, M, P, J, E",✘,True,False
33,Tesla,legal_proceedings,200 chars,201 chars,✘,see (a),see (a)
37,Walmart,legal_proceedings,"5,390 chars","139,663 chars",✘,see (a),see (a)


In [19]:
bonus_df

,company,field,regex,bs4,agree,regex correct,bs4 correct
0,Alphabet,fiscal_year_end,2025-12-31,2025-12-31,✔,True,True
1,Alphabet,legal_proceedings,246 chars,246 chars,✔,see (a),see (a)
2,Alphabet,signature_dates,2026-02-04,2026-02-04,✔,True,True
3,Alphabet,signers,"Sundar Pichai, Anat Ashkenazi, Amie Thuener O'Toole, Frances H. Arnold, Sergey Brin, R. Martin Chávez, L. John Doerr, Roger W. Ferguson Jr., John L. Hennessy, Larry Page, K. Ram Shriram, Robin L. Washington","S, A, F, R. M, L. J, R, J, L, K. R",✘,True,False
4,Amazon,fiscal_year_end,2025-12-31,2025-12-31,✔,True,True
5,Amazon,legal_proceedings,130 chars,130 chars,✔,see (a),see (a)
6,Amazon,signature_dates,2026-02-05,2026-02-05,✔,True,True
7,Amazon,signers,"Andrew R. Jassy, Brian T. Olsavsky, Shelley L. Reynolds, Jeffrey P. Bezos, Keith B. Alexander, Edith W. Cooper, Jamie S. Gorelick, Daniel P. Huttenlocher, Andrew Y. Ng, Indra K. Nooyi, Jonathan J. Rubinstein, Brad D. Smith, Patricia Q. Stonesifer, Wendell P. Weeks","Andrew R. Jassy, Brian T. Olsavsky, Shelley L. Reynolds, Jeffrey P. Bezos, Keith B. Alexander, Edith W. Cooper, Jamie S. Gorelick, Daniel P. Huttenlocher, Andrew Y. Ng, Indra K. Nooyi, Jonathan J. Rubinstein, Brad D. Smith, Patricia Q. Stonesifer, Wendell P. Weeks",✔,True,True
8,Apple,fiscal_year_end,2025-09-27,2025-09-27,✔,True,True
9,Apple,legal_proceedings,"5,372 chars","5,372 chars",✔,see (a),see (a)


### Where do the two versions disagree, and which one is right?

Both versions agree on all fiscal-year-end dates and on most other values. Using the values we read
in the filings as the ground truth, the 7 disagreements are:

| Company / field | What happens with Beautiful Soup | Which is right |
|---|---|---|
| Alphabet – signers | The names are written in small capitals: `S<span>UNDAR</span> P<span>ICHAI</span>`. `get_text("\n")` puts a newline between every piece of text, so the name becomes "S", "UNDAR", "P", "ICHAI" on separate lines and only "S" is found after `/S/`. | regex |
| Microsoft – signers and signature dates | The heading is `<span>SIGNAT</span><span>URES</span>`, which becomes "SIGNAT\nURES". Only the "Signatures" line of the table of contents matches, so the "signature section" is almost the whole report (with many exhibit dates), and the small-caps names are split as for Alphabet. | regex |
| Walmart – Item 3 | In the text, cross-references such as *see "Item 3. Legal Proceedings"* are links (`<a>`). Because of the newline separator they now start a new line and look like a heading. The last one is in the financial statements, and no "Item 4" follows it, so the extracted section runs to the end of the report (139,663 characters). | regex |
| Coca-Cola, Meta, Tesla – Item 3 | Same section, same start and end; the only differences are spaces where inline tags were: "$ 3.3 billion", "( In re Facebook", "Contingencies , to". | regex (cleaner text), content is the same |

**Conclusion.** Beautiful Soup is the safer tool for parsing HTML in general (it handles broken
HTML, comments and attributes correctly) and was fast here as well. But the separator in `get_text`
matters: a newline (or space) between all text pieces breaks words that are split across tags, and
with no separator (`get_text("")`) words from neighbouring table cells or paragraphs would be glued
together. Our regex version avoids both problems because it treats block tags (newline), table cells
(` | `) and inline tags (nothing) differently. For these ten filings the regex version is correct in
every case where the two disagree.

## AI Appendix

**Tool used:** Claude (Anthropic), through Claude Code.

**What it was used for:** downloading the ten 10-K filings from EDGAR, planning the notebook structure,
drafting the cleaning, spell-checking, extraction, verification and Beautiful Soup code, and drafting
the explanatory text.

**What we did ourselves:** _(team: describe honestly — e.g. which parts you checked against the
filings in the browser, which code you rewrote, which values in the check table you re-read yourselves.)_

We have read, run and tested all code in this notebook and can explain every step.